In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"


In [3]:
#基础统计验证

import json
import chromadb
import pandas as pd

CHROMA_PATH = "../data/processed/chroma_db"
COLLECTION_NAME = "medrag_chunks"

# 连接已持久化的索引，不重新建 
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(COLLECTION_NAME)

count = collection.count()
print(f"索引中向量总数: {count}")

#读取之前保存的统计信息，交叉验证 
with open("../data/processed/chroma_stats.json", encoding="utf-8") as f:
    stats = json.load(f)
print(f"\n建索引时记录的统计信息:")
print(json.dumps(stats, ensure_ascii=False, indent=2))

assert count == stats["total_chunks"], "当前索引数量和建索引时的记录对不上，需要检查"
print(f"\n✓ 数量一致性检查通过")

#  随机抽5条，看完整记录（向量+文本+元数据）是否齐全 
sample = collection.get(
    limit=5,
    include=["embeddings", "documents", "metadatas"],
)

print(f"\n=== 抽样检查（5条）===")
for i in range(len(sample["ids"])):
    print(f"\n[{i+1}] id: {sample['ids'][i]}")
    print(f"    向量维度: {len(sample['embeddings'][i])}")
    print(f"    元数据: {sample['metadatas'][i]}")
    print(f"    文本片段: {sample['documents'][i][:100]}...")


索引中向量总数: 170237

建索引时记录的统计信息:
{
  "collection_name": "medrag_chunks",
  "total_chunks": 170237,
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "embedding_dimension": 384,
  "index_built_at": "2026-07-16T11:09:54.869151",
  "chunk_size_stats": {
    "mean": 288.4,
    "max": 450,
    "min": 4
  },
  "metadata_fields": [
    "doc_id",
    "chunk_index",
    "total_chunks",
    "source_title",
    "token_count",
    "pmid",
    "journal",
    "pub_date"
  ]
}

✓ 数量一致性检查通过

=== 抽样检查（5条）===

[1] id: PMC524367
    向量维度: 384
    元数据: {'pmid': '15462679', 'total_chunks': 1, 'doc_id': 'PMC524367', 'journal': 'The International Journal of Behavioral Nutrition and Physical Activity', 'chunk_index': 0, 'token_count': 296, 'source_title': 'Perceived personal, social and environmental barriers to weight maintenance among young women: A community survey', 'pub_date': '2004-10-5'}
    文本片段: Perceived personal, social and environmental barriers to weight maintenance among young women: A com...

[2] i

In [5]:
import os
os.environ["HF_HUB_OFFLINE"]= "1"

from FlagEmbedding import FlagModel

QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages:"

model = FlagModel(
    "BAAI/bge-small-en-v1.5",
    query_instruction_for_retrieval= QUERY_INSTRUCTION,
    use_fp16=True,
    devices=["mps"],
    batch_size=16,
)

print("模型加载完成")
      

/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1449.38it/s]

模型加载完成


In [7]:
# 相似性检索验证（自相似性测试）

import random

self_check_sample = collection.get(limit=200, include=["documents"])
random.seed(42)
check_indices = random.sample(range(len(self_check_sample["ids"])), 10)

pass_count = 0
for idx in check_indices:
    target_id = self_check_sample["ids"][idx]
    target_text = self_check_sample["documents"][idx]

    query_embedding = model.encode_queries([target_text])[0].tolist()
    result = collection.query(query_embeddings=[query_embedding], n_results=1)

    top_id = result["ids"][0][0]
    top_distance = result["distances"][0][0]
    is_self = (top_id == target_id)
    pass_count += is_self

    status = "✓" if is_self else "✗"
    print(f"{status} 查询id={target_id} -> 检索排第一的id={top_id}, 距离={top_distance:.4f}")

print(f"\n自相似性测试通过率: {pass_count}/10")


✓ 查询id=PMC555945_chunk1 -> 检索排第一的id=PMC555945_chunk1, 距离=0.0053
✓ 查询id=PMC548518_chunk0 -> 检索排第一的id=PMC548518_chunk0, 距离=0.0170
✓ 查询id=PMC524373_chunk0 -> 检索排第一的id=PMC524373_chunk0, 距离=0.0108
✓ 查询id=PMC548280 -> 检索排第一的id=PMC548280, 距离=0.0108
✓ 查询id=PMC544929 -> 检索排第一的id=PMC544929, 距离=0.0101
✓ 查询id=PMC514717_chunk0 -> 检索排第一的id=PMC514717_chunk0, 距离=0.0058
✓ 查询id=PMC529469_chunk1 -> 检索排第一的id=PMC529469_chunk1, 距离=0.0077
✓ 查询id=PMC555561 -> 检索排第一的id=PMC555561, 距离=0.0058
✓ 查询id=PMC434149 -> 检索排第一的id=PMC434149, 距离=0.0194
✓ 查询id=PMC519025 -> 检索排第一的id=PMC519025, 距离=0.0192

自相似性测试通过率: 10/10


In [19]:
# query函数定义（跟 03_embed_and_index.py 脚本里的一致）

def query_index(collection, model, query_text, n_results=5, where_filter=None, max_safe_results=100):
    """
    ...（文档字符串不变）...
    """
    if n_results > max_safe_results:
        print(f"提示: 请求的n_results({n_results})超过安全上限({max_safe_results})，已自动调整为{max_safe_results}")
        n_results = max_safe_results

    query_embedding = model.encode_queries([query_text])[0].tolist()
    return collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where_filter,
    )



In [21]:
# 边界情况验证：空查询、超长查询

#测试1：空查询 
print("=== 测试1：空字符串查询 ===")
try:
    empty_result = query_index(collection, model, "", n_results=3)
    print(f"未报错，返回结果数: {len(empty_result['ids'][0])}")
except Exception as e:
    print(f"报错: {type(e).__name__}: {e}")

# 测试2：只有空格的查询 
print("\n=== 测试2：纯空格查询 ===")
try:
    blank_result = query_index(collection, model, "   ", n_results=3)
    print(f"未报错，返回结果数: {len(blank_result['ids'][0])}")
except Exception as e:
    print(f"报错: {type(e).__name__}: {e}")

#  测试3：超长查询（远超模型512 token上限）
print("\n=== 测试3：超长查询 ===")
long_query = "aspirin cardiovascular disease treatment prevention " * 300  # 远超512 token
try:
    long_result = query_index(collection, model, long_query, n_results=3)
    print(f"未报错，返回结果数: {len(long_result['ids'][0])}")
    print(f"排第一的标题: {long_result['metadatas'][0][0]['source_title']}")
except Exception as e:
    print(f"报错: {type(e).__name__}: {e}")

# 测试4：n_results超过索引总量 
print("\n=== 测试4：请求结果数超过索引总量 ===")
try:
    over_result = query_index(collection, model, "aspirin", n_results=999999)
    print(f"未报错，返回结果数: {len(over_result['ids'][0])}")
except Exception as e:
    print(f"报错: {type(e).__name__}: {e}")


=== 测试1：空字符串查询 ===
未报错，返回结果数: 3

=== 测试2：纯空格查询 ===
未报错，返回结果数: 3

=== 测试3：超长查询 ===
未报错，返回结果数: 3
排第一的标题: Lack of benefits for prevention of cardiovascular disease with aspirin therapy in type 2 diabetic patients - a longitudinal observational study

=== 测试4：请求结果数超过索引总量 ===
提示: 请求的n_results(999999)超过安全上限(100)，已自动调整为100
未报错，返回结果数: 100


In [23]:
# 元数据过滤功能验证

#  测试1：按journal过滤 
sample_meta = collection.get(limit=1, include=["metadatas"])
target_journal = sample_meta["metadatas"][0]["journal"]
print(f"=== 测试1：按journal过滤（{target_journal}）===")
filtered_result = query_index(
    collection, model, "disease treatment",
    n_results=5,
    where_filter={"journal": target_journal},
)
print(f"返回结果数: {len(filtered_result['ids'][0])}")
for meta in filtered_result["metadatas"][0]:
    assert meta["journal"] == target_journal, "过滤失效，返回了不匹配的journal"
    print(f"  journal: {meta['journal']} ✓")

# 测试2：对比"加过滤"vs"不加过滤"，结果应该不同 
print("\n=== 测试2：对比有无过滤条件 ===")
no_filter_result = query_index(collection, model, "disease treatment", n_results=5)
print(f"不加过滤，返回journal: {[m['journal'] for m in no_filter_result['metadatas'][0]]}")
print(f"加过滤后，返回journal: {[m['journal'] for m in filtered_result['metadatas'][0]]}")

#  测试3：过滤条件命中0条的情况（不存在的journal）
print("\n=== 测试3：过滤条件不存在时 ===")
empty_filter_result = query_index(
    collection, model, "disease treatment",
    n_results=5,
    where_filter={"journal": "这个期刊名字不存在_测试用"},
)
print(f"返回结果数: {len(empty_filter_result['ids'][0])}（预期0）")


=== 测试1：按journal过滤（The International Journal of Behavioral Nutrition and Physical Activity）===
返回结果数: 5
  journal: The International Journal of Behavioral Nutrition and Physical Activity ✓
  journal: The International Journal of Behavioral Nutrition and Physical Activity ✓
  journal: The International Journal of Behavioral Nutrition and Physical Activity ✓
  journal: The International Journal of Behavioral Nutrition and Physical Activity ✓
  journal: The International Journal of Behavioral Nutrition and Physical Activity ✓

=== 测试2：对比有无过滤条件 ===
不加过滤，返回journal: ['BMC Cancer', 'BMC Health Services Research', 'BMC Pediatrics', 'BMC Public Health', 'BMC Health Services Research']
加过滤后，返回journal: ['The International Journal of Behavioral Nutrition and Physical Activity', 'The International Journal of Behavioral Nutrition and Physical Activity', 'The International Journal of Behavioral Nutrition and Physical Activity', 'The International Journal of Behavioral Nutrition and Physical Activity'